### Middleware

Middleware provides a way to more tightly control what happens inside the agent. Middleware is useful for the following:

· Tracking agent behavior with logging, analytics, and debugging.

. Transforming prompts, tool selection, and output formatting.

. Adding retries, fallbacks, and early termination logic.

. Applying rate limits, guardrails, and PII detection.

In [60]:
import os
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")
model=init_chat_model("groq:qwen/qwen3-32b")

### Summarization of MiddleWare

In [61]:
from langgraph import checkpoint
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage,SystemMessage

agent=create_agent(
    model,
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model,
            trigger=("messages",10),
            keep=("messages",4)
        )
    ]
)

In [62]:
config={"configurable":{"thread_id":"test-1"}}

In [63]:
questions=[
    "what is 2+2",
    "what is 2*2",
    "what is 2-2",
    "what is 2/2",
    "what is 24+2",
    "what is 2+29"
]

for q in questions:
    response=agent.invoke({"messages":[HumanMessage(content=q)]},config)
    print(f"messages:{response}")
    print(f"messages:{len(response['messages'])}")

messages:{'messages': [HumanMessage(content='what is 2+2', additional_kwargs={}, response_metadata={}, id='e3356696-8911-4877-acb5-933fd6183973'), AIMessage(content="<think>\nOkay, so I need to figure out what 2 plus 2 is. Hmm, let me start by recalling basic addition. When you add two numbers together, you're combining their values. So 2 plus 2 would mean starting with two and then adding another two. Let me visualize this. If I have two apples and then I get two more apples, how many apples do I have in total? That should be four apples. \n\nWait, maybe I can use my fingers to count. Let's see, two fingers on one hand and two on the other. If I put them together, that's 1, 2, 3, 4 fingers. Yeah, that seems right. But maybe I should check another way. Let's think about numbers in terms of groups. If there are two groups each containing two items, like two pencils and another two pencils, combining them would give four pencils. \n\nI remember learning addition in school, and the basic 

### Token Size

In [64]:
import os
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")
model=init_chat_model("groq:qwen/qwen3-32b")

In [65]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

@tool
def search_hotels(city: str) -> str:
  """Search hotels - returns long response to use more tokens."""
  return f"""Hotels in {city}:
  1. Grand Hotel - 5 star, $350/night, spa, pool, gym
  2. City Inn - 4 star, $180/night, business center
  3. Budget Stay - 3 star, $75/night, free wifi"""

agent=create_agent(
    model,
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model,
            trigger=("tokens",50),
            keep=("tokens",10)
        )
    ]
)

config={"configurable":{"thread_id":"test-1"}}

def count_tokens(messages):
    total_chars=sum(len(str(m.content))for m in messages)
    return total_chars//4

In [66]:
cities=["paris","london","tokyo","new york","dubai","singapore"]

for city in cities:
    response=agent.invoke(
        {"messages":[HumanMessage(content=f"find hotels in {city}")]},
        config=config
    )

tokens = count_tokens(response["messages"])
print(f"{city}: ~{tokens} tokens, {len(response['messages'])}messages")
print(f"{(response["messages"])}")

KeyboardInterrupt: 

### human in the loop MiddleWare

Pause agent execution for human approval, editing, or rejection of tool calls before they execute. Human-in-the-loop is useful for the
following:

. High-stakes operations requiring human approval (e.g. database writes, financial transactions).  

. Compliance workflows where human oversight is mandatory. 

. Long-running conversations where human feedback guides the agent. 

In [67]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver

def read_email_tool(email_id: str) -> str:
  """Mock function to read an email by its ID."""
  return f"Email content for ID: {email_id}"

def send_email_tool(recipient: str, subject: str, body: str) -> str:
  """Mock function to send an email."""
  return f"Email sent to {recipient} with subject '{subject}'"



In [68]:
agent=create_agent(
    model,
    tools=[read_email_tool,send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool":{
                    "allowed_decisions":["approve","edit","reject"]
                },
                "read_email_tool":False,
            }

        )
    ]
)

In [69]:
config={"configurable":{"thread_id":"test_approve"}} # for reject use test_reject

result=agent.invoke(
    {"messages":[HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'")]},
    config=config
)

In [70]:
result

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='fd5e0bcc-fb82-4df3-9145-126b7ff68694'),
  AIMessage(content='', additional_kwargs={'reasoning_content': "Okay, the user wants to send an email to john@test.com with the subject 'Hello' and body 'How are you?'. Let me check the available tools. There's the send_email_tool which requires recipient, subject, and body. All three are required. The parameters are all strings. The user provided all three pieces of information: recipient is john@test.com, subject is Hello, body is How are you? So I need to call send_email_tool with those arguments. No need to use read_email_tool here since the user isn't asking to read an email. Just format the tool_call correctly with the parameters.\n", 'tool_calls': [{'id': 't0hjyc12t', 'function': {'arguments': '{"body":"How are you?","recipient":"john@test.com","subject":"Hello"}', 'name': 'send_ema

In [71]:
from langgraph.types import Command

if "__interrupt__" in result:
    print("paused Approving. . .")

    result=agent.invoke(
        Command(
            resume={
                "decisions":[
                    {"type":"approve"} # for reject use "reject"
                ]
            }
        ),
        config=config
    )
    
    print(f"Result: {result['messages'][-1].content}")

paused Approving. . .
Result: The email has been successfully sent to john@test.com with the subject "Hello" and the message "How are you?". Let me know if there's anything else you need!


In [72]:
result


{'messages': [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='fd5e0bcc-fb82-4df3-9145-126b7ff68694'),
  AIMessage(content='', additional_kwargs={'reasoning_content': "Okay, the user wants to send an email to john@test.com with the subject 'Hello' and body 'How are you?'. Let me check the available tools. There's the send_email_tool which requires recipient, subject, and body. All three are required. The parameters are all strings. The user provided all three pieces of information: recipient is john@test.com, subject is Hello, body is How are you? So I need to call send_email_tool with those arguments. No need to use read_email_tool here since the user isn't asking to read an email. Just format the tool_call correctly with the parameters.\n", 'tool_calls': [{'id': 't0hjyc12t', 'function': {'arguments': '{"body":"How are you?","recipient":"john@test.com","subject":"Hello"}', 'name': 'send_ema

### Edit

In [75]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver

def read_email_tool(email_id: str) -> str:
  """Mock function to read an email by its ID."""
  return f"Email content for ID: {email_id}"

def send_email_tool(recipient: str, subject: str, body: str) -> str:
  """Mock function to send an email."""
  return f"Email sent to {recipient} with subject '{subject}'"



In [76]:
agent=create_agent(
    model,
    tools=[read_email_tool,send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool":{
                    "allowed_decisions":["approve","edit","reject"]
                },
                "read_email_tool":False,
            }

        )
    ]
)

In [77]:
config={"configurable":{"thread_id":"test_edit"}} # for reject use test_reject

result=agent.invoke(
    {"messages":[HumanMessage(content="Send email to wrong@email.com with subject 'Test' and body 'hello'?'")]},
    config=config
)

In [78]:
result

{'messages': [HumanMessage(content="Send email to wrong@email.com with subject 'Test' and body 'hello'?'", additional_kwargs={}, response_metadata={}, id='0e3bf8b0-9c47-466d-9a9d-fa710ffa042b'),
  AIMessage(content='', additional_kwargs={'reasoning_content': "Okay, let me see. The user wants to send an email to wrong@email.com with the subject 'Test' and body 'hello'. I need to check which tool to use here. The available tools are read_email_tool and send_email_tool. Since the user is asking to send an email, the send_email_tool is the right choice.\n\nLooking at the parameters required for send_email_tool: recipient, subject, and body. The user provided all three: recipient is wrong@email.com, subject is 'Test', and body is 'hello'. So I need to structure the arguments correctly. The function requires these as strings, so I'll make sure to include them as such. Let me double-check the function signature to confirm. Yes, the send_email_tool's parameters are all strings and required. No

In [79]:

if "__interrupt__" in result:
    print("paused Approving. . .")

    result=agent.invoke(
        Command(
            resume={
                "decisions":[
                    {
                        "type":"edit",
                        "edited_action":{
                            "name":"send_email_tool",
                            "args":{
                                "recipient":"correct@gmail.com",
                                "subject":"correct subject",
                                "body":"This was edit by human before sending"
                            }

                        }
                    }
                ]
            }
        ),
        config=config
    )
    
    print(f"Result: {result['messages'][-1].content}")

paused Approving. . .
Result: The email has been successfully sent to **correct@gmail.com** with the subject **"correct subject"**. 

Would you like to make any other changes or send another email?
